# Dimensionality reduction of LDLC-GKP codes — examples

This notebook **generates** dimension-reduced GKP codes derived from classical
low-density lattice codes (LDLCs), **saves** them to a directory, then **loads**
them and **plots** their logical distances against the number of modes.

For each number of modes `n`, the pipeline (in the `SymplecticGKP` package):

1. builds a random `d`-regular classical LDLC and its trivial GKP generator
   `M = √d · H`;
2. keeps it only if its symplectic Gram matrix reduces to a **single logical
   qubit** (invariant factors `(1,…,1,2ℓ)`);
3. divides the reduced symplectic pair to obtain a determinant-`±2` (one-qubit)
   generator;
4. computes the logical distances by solving closest-vector problems, and labels
   them by size so that **`dX ≤ dZ ≤ dY`**.

Each instance is saved as `reduced_ldlc_gkp_n_<n>_<id>.jld2` holding the initial
classical lattice (`classical_generator`), the reduction data (`C`, `k1`, `k2`,
and the added generators `mu_tilde`, `nu_tilde`), the `qubit_generator`, the
LLL-reduced `qubit_generator_lll`, the logicals `XL/YL/ZL`, and their distances
`dX/dY/dZ`. Load one with `SymplecticGKP.load_instance(path)` for the full
structured record; here we only read the distances, so we use plain `JLD2`.

## Producing the data from a terminal

Generation is **slow** — it is dominated by LatticeDecoder's LDLC 4-cycle (loop)
removal, whose per-attempt convergence is very low, so each candidate LDLC takes
on the order of **minutes**, and only some are single-qubit-reducible. Expect a
full run (e.g. 10 instances for each of modes 15–20) to take **many hours**. It
is therefore best launched from a terminal, not from the notebook:

```bash
cd /path/to/SymplecticGKP
nohup julia --project=. scripts/generate_worker.jl \
  --n 15:20 --d 4 --instances 10 --attempts 5000 \
  --seed 0 --outdir data/generated --reduction kz \
  > gen_15_20.log 2>&1 &
```

Watch progress with `tail -f gen_15_20.log`. On a multi-core machine it is much
faster to run one process per number of modes (files don't collide — names are
per-`n`):

```bash
cd /path/to/SymplecticGKP
for n in 15 16 17 18 19 20; do
  nohup julia --project=. scripts/generate_worker.jl \
    --n $n --d 4 --instances 10 --attempts 5000 \
    --seed $n --outdir data/generated --reduction kz \
    > gen_n$n.log 2>&1 &
done
```

Flags: `--instances` is the target per number of modes; `--attempts` is only an
upper bound on tries per `n` (generation stops as soon as the target is reached).
If a run is interrupted, instances saved before their distances were computed
carry `distances_status = "skipped"` and can be completed later with
`--distances-only --n 15:20 --outdir data/generated`.

The generation cell below can launch the **same** run from the notebook
(set `RUN_GENERATION = true`), but for long runs the terminal form is preferred.

## Environment

This notebook uses the lightweight `examples/` environment (`JLD2`, `Plots`) — it
only reads distances and plots, so it does **not** load the full pipeline (no
Oscar/Nemo), which keeps it fast. Instantiate it once:

```bash
julia --project=examples -e 'using Pkg; Pkg.instantiate()'
```

Data *generation* shells out to the package environment (`--project=.`), which
has the full pipeline.


## 1. Configuration & environment

In [ ]:
import Pkg
Pkg.activate(@__DIR__)                 # use the lightweight examples/ environment
using JLD2, Plots, Measures, LaTeXStrings
gr()

# Paths (edit if your layout differs). In IJulia, @__DIR__ is the notebook folder.
const EXAMPLES_DIR = @__DIR__
const PKG_DIR      = dirname(EXAMPLES_DIR)                    # SymplecticGKP package root
const OUTDIR       = joinpath(PKG_DIR, "data", "generated") # where instances are saved
mkpath(OUTDIR)

# Generation parameters
const MODES     = 15:20     # numbers of modes to sweep
const DEGREE    = 4         # classical LDLC degree d
const PER_MODE  = 10        # instances to find per number of modes
const ATTEMPTS  = 5000      # upper bound on tries per n (stops at PER_MODE)
const REDUCTION = "kz"      # CVP pre-reduction: "kz" or "lll"

(; EXAMPLES_DIR, PKG_DIR, OUTDIR)


## 2. Generate the instances

See the note at the top for the terminal command (recommended for long runs).

In [ ]:
# Generation is SLOW (see the note at the top of the notebook). For long runs,
# prefer the terminal command. Set RUN_GENERATION = true to launch it here
# (this blocks the notebook until all instances are found).
RUN_GENERATION = false

if RUN_GENERATION
    worker = joinpath(PKG_DIR, "scripts", "generate_worker.jl")
    nspec  = "$(first(MODES)):$(last(MODES))"
    run(`julia --project=$PKG_DIR $worker --n $nspec --d $DEGREE
         --instances $PER_MODE --attempts $ATTEMPTS --seed 0
         --outdir $OUTDIR --reduction $REDUCTION`)
else
    println("RUN_GENERATION = false — skipping generation.\n",
            "Plotting whatever is already in:\n  ", OUTDIR)
end


## 3. Load the saved distances

In [ ]:
# Load the (completed) instances: read dX/dY/dZ directly from the JLD2 files.
pattern = r"^reduced_ldlc_gkp_n_(\d+)_(\d+)\.jld2$"
n_values = Int[]; dX_values = Float64[]; dY_values = Float64[]; dZ_values = Float64[]; ids = Int[]

for file in sort(readdir(OUTDIR))
    m = match(pattern, file); m === nothing && continue
    vals = JLD2.jldopen(joinpath(OUTDIR, file), "r") do io
        status = haskey(io, "distances_status") ? io["distances_status"] : "ok"
        status == "ok" ? (io["dX"], io["dY"], io["dZ"]) : nothing
    end
    vals === nothing && continue        # skip instances whose distances aren't computed yet
    push!(n_values, parse(Int, m.captures[1]))
    push!(dX_values, vals[1]); push!(dY_values, vals[2]); push!(dZ_values, vals[3])
    push!(ids, parse(Int, m.captures[2]))
end

isempty(n_values) && error("No completed instances found in $OUTDIR — generate some first.")
println("Loaded $(length(n_values)) instance(s); per number of modes: ",
        [(n, count(==(n), n_values)) for n in sort(unique(n_values))])


## 4. Plot the distances vs. number of modes

Red circles `ΔX`, blue squares `ΔY`, green diamonds `ΔZ` (by construction `dX ≤ dZ ≤ dY`). Horizontal lines are the square- and hexagonal-surface-code distances for reference. The figure is also saved as `examples/distance_plot.png`.

In [ ]:
# Staggered x-position for each instance within its number-of-modes group.
stagger = 0.9 / 30
x_positions = zeros(Float64, length(n_values))
for n in unique(n_values)
    idxs = findall(==(n), n_values); c = length(idxs)
    for (i, idx) in enumerate(idxs)
        x_positions[idx] = n + (i - (c + 0.5) / 2) * stagger
    end
end

Plots.default(size=(1400, 600), margins=10mm)
plt = scatter(x_positions, dX_values; marker=:circle, color=:red, label=L"$\Delta_X$",
              legend=:topleft, legendfontsize=12, labelfontsize=12, tickfontsize=12,
              left_margin=10mm, bottom_margin=10mm, right_margin=5mm, top_margin=5mm)
scatter!(plt, x_positions, dY_values; marker=:square,  color=:blue,  label=L"$\Delta_Y$")
scatter!(plt, x_positions, dZ_values; marker=:diamond, color=:green, label=L"$\Delta_Z$")

# light gridline per instance, bold separators between groups
vline!(plt, x_positions; color=:gray, alpha=0.3, label="")
vline!(plt, [n + 0.5 for n in minimum(n_values):(maximum(n_values) - 1)];
       color=:black, alpha=0.9, label="")

# surface-code distance references (plain lattice norm, same convention as the data):
# distance-3 patch (n=9) and distance-5 patch (n=25), square & hexagonal single-mode GKP.
sq9  = sqrt(3 / 2);   hex9  = 3^(1 / 4)             # 9-mode:  √(3/2),  3^(1/4)
sq25 = sqrt(5 / 2);   hex25 = sqrt(5 / sqrt(3))     # 25-mode: √(5/2),  √5·3^(-1/4)
hline!(plt, [sq9];   color=:red,  linestyle=:solid,   label="Sq-SC (n=9)")
hline!(plt, [sq25];  color=:red,  linestyle=:dashdot, label="Sq-SC (n=25)")
hline!(plt, [hex9];  color=:blue, linestyle=:dash,    label="Hex-SC (n=9)")
hline!(plt, [hex25]; color=:blue, linestyle=:dot,     label="Hex-SC (n=25)")

xlabel!(plt, "number of modes"); ylabel!(plt, "Distance")
xticks!(plt, sort(unique(n_values)))
xlims!(plt, minimum(n_values) - 0.5, maximum(n_values) + 0.5)
allv = vcat(dX_values, dY_values, dZ_values); refs = [sq9, sq25, hex9, hex25]
ylims!(plt, min(minimum(allv), minimum(refs)) * 0.95, max(maximum(allv), maximum(refs)) * 1.05)

savefig(plt, joinpath(EXAMPLES_DIR, "distance_plot.png"))
plt


## 5. Distances sorted within each mode bin

Same staggered scatter as above, but within each number-of-modes bin the instances
are ordered left→right by **increasing minimum distance** (`dX`). The rightmost
point of each bin is therefore the best code (largest minimum distance) for that
number of modes. Saved as `examples/distance_plot_sorted.png`.

In [ ]:
# Same staggered scatter as above, but within each mode bin the instances are
# ordered left→right by increasing minimum distance (dX), so the best code
# (largest minimum distance) sits at the right edge of its bin.
stagger = 0.9 / 30
x_positions = zeros(Float64, length(n_values))
for n in unique(n_values)
    idxs  = findall(==(n), n_values)
    order = sort(idxs; by = i -> dX_values[i])   # smallest min-distance left, largest right
    c = length(order)
    for (i, idx) in enumerate(order)
        x_positions[idx] = n + (i - (c + 0.5) / 2) * stagger
    end
end

Plots.default(size=(1400, 600), margins=10mm)
plt = scatter(x_positions, dX_values; marker=:circle, color=:red, label=L"$\Delta_X$",
              legend=:topleft, legendfontsize=12, labelfontsize=12, tickfontsize=12,
              left_margin=10mm, bottom_margin=10mm, right_margin=5mm, top_margin=5mm)
scatter!(plt, x_positions, dY_values; marker=:square,  color=:blue,  label=L"$\Delta_Y$")
scatter!(plt, x_positions, dZ_values; marker=:diamond, color=:green, label=L"$\Delta_Z$")

# light gridline per instance, bold separators between groups
vline!(plt, x_positions; color=:gray, alpha=0.3, label="")
vline!(plt, [n + 0.5 for n in minimum(n_values):(maximum(n_values) - 1)];
       color=:black, alpha=0.9, label="")

# surface-code distance references (plain lattice norm, same convention as the data):
# distance-3 patch (n=9) and distance-5 patch (n=25), square & hexagonal single-mode GKP.
sq9  = sqrt(3 / 2);   hex9  = 3^(1 / 4)             # 9-mode:  √(3/2),  3^(1/4)
sq25 = sqrt(5 / 2);   hex25 = sqrt(5 / sqrt(3))     # 25-mode: √(5/2),  √5·3^(-1/4)
hline!(plt, [sq9];   color=:red,  linestyle=:solid,   label="Sq-SC (n=9)")
hline!(plt, [sq25];  color=:red,  linestyle=:dashdot, label="Sq-SC (n=25)")
hline!(plt, [hex9];  color=:blue, linestyle=:dash,    label="Hex-SC (n=9)")
hline!(plt, [hex25]; color=:blue, linestyle=:dot,     label="Hex-SC (n=25)")

xlabel!(plt, "number of modes"); ylabel!(plt, "Distance")
xticks!(plt, sort(unique(n_values)))
xlims!(plt, minimum(n_values) - 0.5, maximum(n_values) + 0.5)
allv = vcat(dX_values, dY_values, dZ_values); refs = [sq9, sq25, hex9, hex25]
ylims!(plt, min(minimum(allv), minimum(refs)) * 0.95, max(maximum(allv), maximum(refs)) * 1.05)

savefig(plt, joinpath(EXAMPLES_DIR, "distance_plot_sorted.png"))
plt


## 6. Best code per mode (largest minimum distance) with optional linear fit

For each number of modes we keep only the instance with the **largest minimum
distance** (largest `dX`) and plot its `dX/dY/dZ`. Set `ADD_LINEAR_FIT = true` to
overlay a least-squares linear fit for each distance (dotted lines); set it to
`false` (or comment the fit block) to hide them. Saved as `examples/distance_plot_best.png`.

In [ ]:
# Best code per number of modes = the instance with the largest MINIMUM distance
# (= largest dX, since logicals are labelled so dX ≤ dZ ≤ dY). Optionally overlay
# a linear fit for each of dX/dY/dZ; toggle with ADD_LINEAR_FIT (or comment the
# `if ADD_LINEAR_FIT ... end` block).
ADD_LINEAR_FIT = true   # set false to hide the fitted lines

ns = sort(unique(n_values))
best_dX = Float64[]; best_dY = Float64[]; best_dZ = Float64[]
for n in ns
    idxs = findall(==(n), n_values)
    b    = idxs[argmax(dX_values[idxs])]          # code with the largest minimum distance
    push!(best_dX, dX_values[b]); push!(best_dY, dY_values[b]); push!(best_dZ, dZ_values[b])
end

# simple least-squares line fit (closed form; no LinearAlgebra dependency)
function _linfit(x, y)
    x̄ = sum(x) / length(x); ȳ = sum(y) / length(y)
    slope = sum((x .- x̄) .* (y .- ȳ)) / sum((x .- x̄) .^ 2)
    return ȳ - slope * x̄, slope        # (intercept, slope)
end

Plots.default(size=(1400, 600), margins=10mm)
plt = scatter(ns, best_dX; marker=:circle,  color=:red,   label=L"$\Delta_X$",
              legend=:bottomright, legendfontsize=12, labelfontsize=12, tickfontsize=12,
              left_margin=10mm, bottom_margin=10mm, right_margin=5mm, top_margin=5mm)
scatter!(plt, ns, best_dY; marker=:square,  color=:blue,  label=L"$\Delta_Y$")
scatter!(plt, ns, best_dZ; marker=:diamond, color=:green, label=L"$\Delta_Z$")

if ADD_LINEAR_FIT
    for (ys, col) in ((best_dX, :red), (best_dY, :blue), (best_dZ, :green))
        a, b = _linfit(ns, ys)
        plot!(plt, ns, a .+ b .* ns; color=col, linestyle=:dot, label="")
    end
end

# surface-code distance references (plain lattice norm, same convention as the data):
# distance-3 patch (n=9) and distance-5 patch (n=25), square & hexagonal single-mode GKP.
sq9  = sqrt(3 / 2);   hex9  = 3^(1 / 4)             # 9-mode:  √(3/2),  3^(1/4)
sq25 = sqrt(5 / 2);   hex25 = sqrt(5 / sqrt(3))     # 25-mode: √(5/2),  √5·3^(-1/4)
hline!(plt, [sq9];   color=:red,  linestyle=:solid,   label="Sq-SC (n=9)")
hline!(plt, [sq25];  color=:red,  linestyle=:dashdot, label="Sq-SC (n=25)")
hline!(plt, [hex9];  color=:blue, linestyle=:dash,    label="Hex-SC (n=9)")
hline!(plt, [hex25]; color=:blue, linestyle=:dot,     label="Hex-SC (n=25)")

xlabel!(plt, "number of modes"); ylabel!(plt, "Distance")
xticks!(plt, ns)
allv = vcat(best_dX, best_dY, best_dZ); refs = [sq9, sq25, hex9, hex25]
ylims!(plt, min(minimum(allv), minimum(refs)) * 0.95, max(maximum(allv), maximum(refs)) * 1.05)

savefig(plt, joinpath(EXAMPLES_DIR, "distance_plot_best.png"))
plt
